In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.2 MB/s eta 0:00:00


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640
)

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b4b2ae48c80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [7]:
!zip -r train-4.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/train-2/ (stored 0%)
  adding: content/runs/detect/train-2/labels.jpg (deflated 48%)
  adding: content/runs/detect/train-2/BoxPR_curve.png (deflated 30%)
  adding: content/runs/detect/train-2/confusion_matrix.png (deflated 31%)
  adding: content/runs/detect/train-2/BoxF1_curve.png (deflated 32%)
  adding: content/runs/detect/train-2/confusion_matrix_normalized.png (deflated 30%)
  adding: content/runs/detect/train-2/val_batch0_labels.jpg (deflated 15%)
  adding: content/runs/detect/train-2/results.png (deflated 9%)
  adding: content/runs/detect/train-2/train_batch0.jpg (deflated 26%)
  adding: content/runs/detect/train-2/val_batch0_pred.jpg (deflated 17%)
  adding: content/runs/detect/train-2/train_batch40.jpg (deflated 32%)
  adding: content/runs/detect/train-2/results.csv (deflated 67%)
  adding: content/runs/detect/train-2/weights/ (stored 0%)
  adding: content/runs/detect/tr

In [9]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-4/weights/best.pt")

results = model.predict("/content/drive/MyDrive/CS317/datasets/test/images/1PlateBaza480.jpg")

boxes = []

for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        x_center = float(box.xywh[0][0])  # x center

        char = model.names[cls_id]
        boxes.append((char, x_center))

# sort trái → phải
boxes = sorted(boxes, key=lambda x: x[1])

plate = "".join([b[0] for b in boxes])

print("Plate:", plate)


image 1/1 /content/drive/MyDrive/CS317/datasets/test/images/1PlateBaza480.jpg: 160x640 1 5, 2 8s, 1 G, 1 K, 1 P, 1 Z, 11.0ms
Speed: 1.2ms preprocess, 11.0ms inference, 1.5ms postprocess per image at shape (1, 3, 160, 640)
Plate: ZG885PK


In [10]:
import os
from ultralytics import YOLO

# ===== PATH =====
MODEL_PATH = "/content/runs/detect/train-4/weights/best.pt"
IMG_DIR = "/content/drive/MyDrive/CS317/datasets/test/images"
LBL_DIR = "/content/drive/MyDrive/CS317/datasets/test/labels"

model = YOLO(MODEL_PATH)

# ===== Hàm convert YOLO label → string =====
def label_to_string(label_path, names):
    chars = []

    with open(label_path) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            char = names[int(cls)]
            chars.append((char, x))  # x center

    chars = sorted(chars, key=lambda x: x[1])
    return "".join([c[0] for c in chars])

# ===== Hàm predict → string =====
def predict_to_string(result, names):
    chars = []

    for box in result.boxes:
        cls_id = int(box.cls[0])
        x_center = float(box.xywh[0][0])
        char = names[cls_id]
        chars.append((char, x_center))

    chars = sorted(chars, key=lambda x: x[1])
    return "".join([c[0] for c in chars])

# ===== LOOP TEST =====
images = [f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")]

total = 0
correct = 0
char_total = 0
char_correct = 0

for img_name in images:
    img_path = os.path.join(IMG_DIR, img_name)
    lbl_path = os.path.join(LBL_DIR, img_name.replace(".jpg", ".txt"))

    if not os.path.exists(lbl_path):
        continue

    # predict
    results = model.predict(img_path, conf=0.25, verbose=False)
    pred = predict_to_string(results[0], model.names)

    # ground truth
    gt = label_to_string(lbl_path, model.names)

    total += 1

    if pred == gt:
        correct += 1

    # char-level accuracy
    min_len = min(len(pred), len(gt))
    char_total += len(gt)

    for i in range(min_len):
        if pred[i] == gt[i]:
            char_correct += 1

    print(f"{img_name}")
    print(f"GT  : {gt}")
    print(f"PRED: {pred}")
    print("-"*30)

# ===== KẾT QUẢ =====
print("Exact match acc:", correct / total)
print("Char acc:", char_correct / char_total)

3xemay17.jpg
GT  : 56507B253
PRED: 56507B253
------------------------------
2PlateBaza476.jpg
GT  : ZG385SZ
PRED: ZG385SZ
------------------------------
3xemay513.jpg
GT  : 85793Y014
PRED: 85793Y014
------------------------------
3xemay1041.jpg
GT  : 45431U65
PRED: 45431U65
------------------------------
1xemay1524.jpg
GT  : 55403L15
PRED: 55403L15
------------------------------
3xemay511.jpg
GT  : 44730R11
PRED: 44730RH11
------------------------------
2PlateBaza256.jpg
GT  : G22SKD
PRED: G22SKD0
------------------------------
3xemay780.jpg
GT  : 25096Z811
PRED: 25096Z811
------------------------------
3xemay1376.jpg
GT  : 95190X121
PRED: 95190X121
------------------------------
1xemay1289.jpg
GT  : 16065N417
PRED: 186065N417
------------------------------
2xemay326.jpg
GT  : 25096S127
PRED: 25096S127
------------------------------
10xemay47.jpg
GT  : 95093K010
PRED: 95093K010
------------------------------
3CarLongPlate874.jpg
GT  : 51F1555
PRED: 51F15585
----------------------------